In [6]:
pip install biopython

In [7]:
!pip install matplotlib

In [12]:

# =============================
# IMPORTS
# =============================
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.Blast import NCBIWWW, NCBIXML
from collections import Counter
import matplotlib.pyplot as plt
import math
import sys
class Tee:
    def __init__(self, filename):
        self.file = open(filename, "a", encoding="utf-8")
        self.stdout = sys.stdout

    def write(self, message):
        self.stdout.write(message)
        self.stdout.flush()
        self.file.write(message)
        self.file.flush()

    def flush(self):
        self.stdout.flush()
        self.file.flush()


Entrez.email = "your_email@example.com"


# =============================
# FETCH
# =============================
def fetch_ncbi_data(query):

    handle = Entrez.esearch(db="nucleotide", term=query, retmax=20)
    record = Entrez.read(handle)
    handle.close()

    if not record["IdList"]:
        print(" No result found.")
        return None

    for seq_id in record["IdList"]:
        try:
            handle = Entrez.efetch(
                db="nucleotide",
                id=seq_id,
                rettype="gb",
                retmode="text"
            )

            seq_record = SeqIO.read(handle, "genbank")
            handle.close()

            dna = str(seq_record.seq)

            if dna and len(dna) > 50:
                return seq_record

        except:
            continue

    print(" No valid DNA found.")
    return None


# =============================
# BASIC
# =============================
def nucleotide_count(dna):
    return {
        "A": dna.count("A"),
        "T": dna.count("T"),
        "G": dna.count("G"),
        "C": dna.count("C")
    }


def gc_content(dna):
    return round((dna.count("G") + dna.count("C")) / len(dna) * 100, 2)


def ta_content(dna):
    return round((dna.count("T") + dna.count("A")) / len(dna) * 100, 2)


# =============================
# PROCESS
# =============================
def process_sequence(dna):

    seq = Seq(dna)
    usable_len = len(dna) - (len(dna) % 3)

    protein = str(seq[:usable_len].translate())

    return {
        "Complement": str(seq.complement())[:100],
        "Reverse": dna[::-1][:100],
        "Reverse Complement": str(seq.reverse_complement())[:100],
        "Protein": protein[:100]
    }


# =============================
# K-MERS
# =============================
def top_kmers(dna, k=3):

    kmers = [dna[i:i+k] for i in range(len(dna)-k+1)]
    counts = Counter(kmers)

    return counts.most_common(5)


# =============================
# ORFs
# =============================
def find_orfs(dna):

    start = "ATG"
    stops = ["TAA", "TAG", "TGA"]

    orfs = []

    for i in range(len(dna)-2):

        if dna[i:i+3] == start:

            for j in range(i+3, len(dna)-2, 3):

                codon = dna[j:j+3]

                if codon in stops:
                    orf = dna[i:j+3]
                    orfs.append(orf)
                    break

    orfs.sort(key=len, reverse=True)

    return orfs[:5]


# =============================
# SIMILARITY
# =============================
def similarity(seq1, seq2):

    min_len = min(len(seq1), len(seq2))

    matches = sum(
        1 for i in range(min_len)
        if seq1[i] == seq2[i]
    )

    return round((matches / min_len) * 100, 2)


# =============================
# BLAST
# =============================
def blast_evalue(dna):

    result_handle = NCBIWWW.qblast("blastn", "nt", dna[:1000])
    blast_record = NCBIXML.read(result_handle)

    if blast_record.alignments:
        return blast_record.alignments[0].hsps[0].expect

    return None


def evalue_score(e):
    if e is None or e == 0:
        return 100
    return -math.log10(e)


# =============================
# MAIN
# =============================
print("🧬 Welcome to Biopylogic Tool")

print("""
1) Single Sequence Analysis
2) Sequence Comparison
3) Phylogenetic Tree
""")

choice = input("Enter choice: ")
sys.stdout = Tee("biopylogic_results.txt")



# =============================
# OPTION 1
# =============================
if choice == "1":

    print("\n🧬 Single Sequence Analysis")

    query = input("Enter gene + organism:(example: rbcL Zea mays)\n> ")

    print("\n🔎 Fetching data from NCBI...")

    record = fetch_ncbi_data(query)

    if record is None:
        exit()

    dna = str(record.seq).upper()

    accession = record.annotations.get("accessions", ["N/A"])[0]

    print("\n🔎 Record Info:")

    print("ID:", record.id)
    print("Name:", record.name)
    print("Locus:", record.name)
    print("Accession:", accession)
    print("Definition:", record.description)
    print("Organism:", record.annotations.get("organism"))
    print("Sequence Length:", len(dna), "bp")

    print("\nSequence (first 100 bp):")
    print(dna[:100])

    print("\n🔬 Basic Analysis:")

    counts = nucleotide_count(dna)

    for base in counts:
        print(base, ":", counts[base])

    print("\nGC Content:", gc_content(dna), "%")
    print("TA Content:", ta_content(dna), "%")

    processed = process_sequence(dna)

    print("\nComplement:")
    print(processed["Complement"])

    print("\nReverse:")
    print(processed["Reverse"])

    print("\nReverse Complement:")
    print(processed["Reverse Complement"])

    print("\n🧬 Protein:")
    print(processed["Protein"])

    print("\n🔬 Top k-mers:")
    for kmer, count in top_kmers(dna):
        print(kmer, ":", count)

    print("\n🧬 ORFs Found (Top 5 longest):")
    orfs = find_orfs(dna)

    for i, orf in enumerate(orfs):
        percent = round(len(orf)/len(dna)*100, 2)

        print(f"ORF {i+1}: {len(orf)} bp ({percent}%)")

    # BLAST
    print("\n🔎 Running BLAST...")

    evalue = blast_evalue(dna)

    print("\n🧬 E-value:", evalue)

    print("\n🧬 E-value Score (−log10):")
    print(evalue_score(evalue))


# =============================
# OPTION 2
# =============================
elif choice == "2":

    print("\n🧬 Sequence Comparison")

    query1 = input("Enter gene + organism:(example: rbcL Ulva lactuca)\n> ")
    query2 = input("Enter gene + organism:(example: rbcL Sargassum latifolium)\n> ")

    record1 = fetch_ncbi_data(query1)
    record2 = fetch_ncbi_data(query2)

    if record1 is None or record2 is None:
        print("❌ Error")
        exit()

    dna1 = str(record1.seq).upper()
    dna2 = str(record2.seq).upper()

    print("\n========== 🧬 SEQ 1 ==========")
    print("Organism:", record1.annotations.get("organism"))
    print("GC:", gc_content(dna1), "%")
    print("TA:", ta_content(dna1), "%")

    p1 = process_sequence(dna1)
    print("Protein:", p1["Protein"][:50])

    print("\n========== 🧬 SEQ 2 ==========")
    print("Organism:", record2.annotations.get("organism"))
    print("GC:", gc_content(dna2), "%")
    print("TA:", ta_content(dna2), "%")

    p2 = process_sequence(dna2)
    print("Protein:", p2["Protein"][:50])

    sim = similarity(dna1, dna2)

    print("\n🔥 Similarity:", sim, "%")

    print("\n🔎 Running BLAST...")

    e1 = blast_evalue(dna1)
    e2 = blast_evalue(dna2)

    score1 = evalue_score(e1)
    score2 = evalue_score(e2)

    print("\n🧬 E-value Seq1:", e1)
    print("🧬 E-value Seq2:", e2)

    print("\n🧬 E-value Score Seq1:", score1)
    print("🧬 E-value Score Seq2:", score2)

    print("\n🧠 Interpretation:")

    if sim > 70 and e1 < 1e-5 and e2 < 1e-5:
        print("Strong biological similarity between sequences")

    elif sim > 50:
        print("Moderate similarity between sequences")

    else:
        print("Low similarity between sequences")

    # Plot
    labels = ["GC", "TA", "Similarity", "E-value"]
    x = range(len(labels))

    plt.figure()

    plt.bar(x, [gc_content(dna1), ta_content(dna1), sim, score1])
    plt.bar([i + 0.4 for i in x],
            [gc_content(dna2), ta_content(dna2), sim, score2])

    plt.xticks([i + 0.2 for i in x], labels)

    plt.title("Sequence Comparison")
    plt.show()

    # =============================
# OPTION 3
# =============================
elif choice == "3":

    print("\n🧬 Phylogenetic Tree")

    gene = input("Enter gene name (example: rbcL)\n> ")

    org1 = input("Enter organism 1:\n> ")
    org2 = input("Enter organism 2:\n> ")
    org3 = input("Enter organism 3:\n> ")

    # Build queries
    query1 = f"{gene} {org1}"
    query2 = f"{gene} {org2}"
    query3 = f"{gene} {org3}"

    print("\nFetching sequences from NCBI...")

    record1 = fetch_ncbi_data(query1)
    record2 = fetch_ncbi_data(query2)
    record3 = fetch_ncbi_data(query3)

    if record1 is None or record2 is None or record3 is None:
        print("❌ Error fetching sequences")
        exit()

    dna1 = str(record1.seq).upper()
    dna2 = str(record2.seq).upper()
    dna3 = str(record3.seq).upper()

    name1 = org1
    name2 = org2
    name3 = org3

    print("\n✅ Sequences fetched successfully!")

    print("\n1)", name1, "| Length:", len(dna1), "bp")
    print("2)", name2, "| Length:", len(dna2), "bp")
    print("3)", name3, "| Length:", len(dna3), "bp")

    # Similarity
    sim12 = similarity(dna1, dna2)
    sim13 = similarity(dna1, dna3)
    sim23 = similarity(dna2, dna3)

    print("\n🔬 Similarity Results:")

    print(f"{name1} vs {name2}: {sim12}%")
    print(f"{name1} vs {name3}: {sim13}%")
    print(f"{name2} vs {name3}: {sim23}%")

    # Distance
    dist12 = round(100 - sim12, 2)
    dist13 = round(100 - sim13, 2)
    dist23 = round(100 - sim23, 2)

    print("\n📏 Distance Matrix:")

    print(f"{name1} ↔ {name2}: {dist12}")
    print(f"{name1} ↔ {name3}: {dist13}")
    print(f"{name2} ↔ {name3}: {dist23}")

    # Simple Tree
    pairs = {
        (name1, name2): sim12,
        (name1, name3): sim13,
        (name2, name3): sim23
    }

    closest_pair = max(pairs, key=pairs.get)

    third_name = [
        x for x in [name1, name2, name3]
        if x not in closest_pair
    ][0]

    first, second = closest_pair

    print("\n🌳 Simple Phylogenetic Tree:\n")

    print(f"        ┌── {first}")
    print("    ┌───┤")
    print(f"    │   └── {second}")
    print("────┤")
    print(f"    └──── {third_name}")

🧬 Welcome to Biopylogic Tool

1) Single Sequence Analysis
2) Sequence Comparison
3) Phylogenetic Tree

Enter choice: 1

🧬 Single Sequence Analysis
Enter gene + organism:(example: rbcL Zea mays)
> rbcL Zea mays

🔎 Fetching data from NCBI...

🔎 Record Info:
ID: PX389917.1
Name: PX389917
Locus: PX389917
Accession: PX389917
Definition: Zea mays isolate DS553 chloroplast, complete genome
Organism: Zea mays
Sequence Length: 140453 bp

Sequence (first 100 bp):
GAAATACCCAATATCCTGTTGGAACAAGATATTGGGTATTTCCGGCTTTCCTTCCTTCAAAAATTCCTATATGTTTAGGAGAAAAACCTTATCCATTAAG

🔬 Basic Analysis:
A : 43317
T : 43151
G : 27086
C : 26899

GC Content: 38.44 %
TA Content: 61.56 %

Complement:
CTTTATGGGTTATAGGACAACCTTGTTCTATAACCCATAAAGGCCGAAAGGAAGGAAGTTTTTAAGGATATACAAATCCTCTTTTTGGAATAGGTAATTC

Reverse:
AAACAAATAAGACTTAGAAAGATAAGACTTAAGTTAATTGCTGCTCTAAATCATAGGAAAGAACATGAAAGTATTGAGCACTTTACGGCTCATCCGTGCT

Reverse Complement:
TTTGTTTATTCTGAATCTTTCTATTCTGAATTCAATTAACGACGAGATTTAGTATCCTTTCTTGTACTTTCATAACTCGTGAAATGCCGAGTAGG